<a href="https://colab.research.google.com/github/krgyaan/LLM-Finetuning/blob/main/Non_Instruction_Pretrain_LLM_finetuning_on_domain_specific_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## LLM Finetuning Library or Framework
1. **Transformers**
2. **PEFT (Parameter-Efficient Fine-Tuning)**
3. **Accelerate**
4. **trl (Transformers Reinforcement Learning)**
5. **Llama Factory**
6. **Unsloth**
7. **FastChat**
8. **DeepSpeed**
9. **Colossal-AI**
10. **OpenLLM**
11. **Axolotl**
12. **SkyPilot**
13. **LightLLM / vLLM**



In [ ]:
!pip install -U transformers peft bitsandbytes accelerate

In [ ]:
!pip install -U trl

In [ ]:
!pip install -U PyMuPDF

## Non Instruction Finetune Dataset. e.g.,

* dataset = load_dataset("HuggingFaceFW/fineweb")
* pubmed = load_dataset("ncbi/pubmed")
* dataset = load_dataset("datajuicer/the-pile-pubmed-abstracts-refined-by-data-juicer")
* dataset = load_dataset("open-llm-leaderboard/open_llm_corpus")
* owt = load_dataset("Skylion007/openwebtext")
* ds = load_dataset("armanc/scientific_papers")

## Instruction Finetuning Dataset. e.g.,

* https://huggingface.co/datasets/Amod/mental_health_counseling_conversations
* https://huggingface.co/datasets/yahma/alpaca-cleaned
* https://huggingface.co/datasets/Open-Orca/OpenOrca
* https://huggingface.co/datasets/tatsu-lab/alpaca
* https://huggingface.co/datasets/OpenAssistant/oasst1

## Preference Finetuning Dataset. e.g.,
* https://huggingface.co/datasets/Anthropic/hh-rlhf
* https://huggingface.co/datasets/argilla/ultrafeedback-binarized-preferences
* https://huggingface.co/datasets/xinlai/Math-Step-DPO-10K

## Prebuilt Data from huggingface data hub.

In [ ]:
from datasets import Dataset, load_dataset

In [ ]:
stories = load_dataset("roneneldan/TinyStories", split="train")

In [ ]:
stories

In [ ]:
stories[0]

## Our own custom data (non instruction data) for domain specific

In [ ]:
import fitz

In [ ]:
def extract_text_from_pdf(pdf_path):
  text_block = []
  with fitz.open(pdf_path) as doc:
    for page in doc:
      text = page.get_text("text").strip()
      if text:
        text_block.append(text)
    return text_block

In [ ]:
pdf_data = '/content/Metformin.pdf'
pdf_text = extract_text_from_pdf(pdf_data)

In [ ]:
pdf_text

In [ ]:
import re
def split_into_chunk(pages):
  paragraphs = []
  for page_text in pages:
    # Split on double line break or long newlines
    chunks = re.split(r'\n\s*\n', page_text)
    for chunk in chunks:
      clean = chunk.strip()
      if len(clean) > 30:
        paragraphs.append(clean)
  return paragraphs

In [ ]:
paragraphs = split_into_chunk(pdf_text)

In [ ]:
paragraphs

In [ ]:
data = [{"text": p} for p in paragraphs]

In [ ]:
data

In [ ]:
dataset = Dataset.from_list(data)

In [ ]:
dataset

## Select Model for training
According to your machine (cabability)

In [ ]:
# The TinyLlama project aims to pretrain a 1.1B Llama model on 3 trillion tokens.
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
# meta-llama/Llama-2-7b-chat-hf
# meta-llama/Llama-3.1-8B

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenizer_fn(examples):
  tokens = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

In [ ]:
tokenized = dataset.map(tokenizer_fn, batched=True, remove_columns=["text"])

In [ ]:
tokenized

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
training_args = TrainingArguments(
    output_dir = "./llm-gem-data",    # Save model/checkpoints -> folder path -> where your model ends up
    num_train_epochs = 2,             # Training cycle -> 1-5 -> More = longer training
    per_device_train_batch_size = 2,  # Batch per GPU -> 1-8(Colab) -> Affects speed & memore
    save_steps = 500,                 # Save frequency -> 100-1000 -> More = frequent checkpoints
    save_total_limit = 2,             # Keep last N ->1-3 -> Saves disks
    logging_steps = 50,               # Log every X step -. 20-100  -> for progress monitoring
    learning_rate = 2e-5,             # Weight updates rate -> 1e-5 - 5e-5 ->Controls stability
    fp16 = True,                      # Mixed precision -> True -> Faster + less memory
    report_to = "none"                # Logging destination -> "none" -> Keeps it simple
)

In [ ]:
from transformers import TrainingArguments
help(TrainingArguments) # check all the present args

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized
)

In [ ]:
trainer.train()
# Will throw Error, because we are trying to re-train the entire model
'''
CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 1.81 MiB is free.
'''

### Here we are not specfiying anything means this is full fine-tuning.
#### Now we have two methods for Partial finetuning
* Feerze some layer and finetune unfreeze layer (old CNN and Bert sytel method)
* LoRA (Append some external weight to the already trained pretrain weight)

## Now let's see the LoRA based method.

In [ ]:
# Clear the cache and Garbage Collection

import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!pip install -U transformers peft accelerate

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, TaskType, get_peft_model
from datasets import load_dataset

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

def tokenizer_fn(examples):
  tokens = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens


data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
tokenized = dataset.map(tokenizer_fn, batched=True, remove_columns=["text"])

In [ ]:
tokenized

In [ ]:
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model.config.use_cache = False          # Required for gradient checkpointing
model.enable_input_require_grads()      # Required for LoRA + quantized models

In [ ]:
from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig, TaskType

model = prepare_model_for_kbit_training(model)   # Must be called before get_peft_model

In [ ]:
# Loaded Quantized Model

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0}        # Force everything onto GPU 0, not "auto"
)

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

In [ ]:
qlora_model = get_peft_model(model, lora_config)
qlora_model.print_trainable_parameters()   # sanity check

In [ ]:
training_args = TrainingArguments(
    output_dir = "./llm-gem-data",    # Save model/checkpoints -> folder path -> where your model ends up
    num_train_epochs = 2,             # Training cycle -> 1-5 -> More = longer training
    per_device_train_batch_size = 2,  # Batch per GPU -> 1-8(Colab) -> Affects speed & memore
    save_steps = 500,                 # Save frequency -> 100-1000 -> More = frequent checkpoints
    save_total_limit = 2,             # Keep last N ->1-3 -> Saves disks
    logging_steps = 50,               # Log every X step -. 20-100  -> for progress monitoring
    learning_rate = 2e-5,             # Weight updates rate -> 1e-5 - 5e-5 ->Controls stability
    fp16 = True,                      # Mixed precision -> True -> Faster + less memory
    report_to = "none",               # Logging destination -> "none" -> Keeps it simple
    optim="paged_adamw_8bit",         # Memory-efficient optimizer for QLoRA
    gradient_checkpointing=True,
)

In [ ]:
trainer = Trainer(
    model=qlora_model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator
)

In [ ]:
trainer.train()

In [ ]:
trained_model_path = "/content/llm-gem-data/checkpoint-4/"

In [ ]:
import os
print(os.listdir('/content'))

In [ ]:
import zipfile
import os

folder_path = "/content/llm-gem-data"
zip_path = "/content/llm-gem-data.zip"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zip_ref:
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, folder_path)
            zip_ref.write(full_path, arcname)

In [ ]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
trained_model = PeftModel.from_pretrained(base_model, trained_model_path)

In [ ]:
prompt="Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"

In [ ]:
inputs=tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = trained_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print("\nModel Output: \n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))